In [2]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

In [4]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [5]:
model_id = "distil-whisper/distil-large-v2"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

config.json: 100%|██████████| 2.29k/2.29k [00:00<00:00, 6.07MB/s]
model.safetensors: 100%|██████████| 1.51G/1.51G [00:18<00:00, 83.8MB/s]
generation_config.json: 100%|██████████| 3.59k/3.59k [00:00<00:00, 12.5MB/s]


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1280)
      (layers): ModuleList(
        (0-31): 32 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_features=5120, out_features=1280, bias=Tru

In [6]:
processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    torch_dtype=torch_dtype,
    device=device,
)

preprocessor_config.json: 100%|██████████| 339/339 [00:00<00:00, 1.24MB/s]
tokenizer_config.json: 100%|██████████| 283k/283k [00:00<00:00, 5.45MB/s]
vocab.json: 100%|██████████| 836k/836k [00:00<00:00, 21.8MB/s]
tokenizer.json: 100%|██████████| 2.48M/2.48M [00:00<00:00, 34.3MB/s]
merges.txt: 100%|██████████| 494k/494k [00:00<00:00, 18.7MB/s]
normalizer.json: 100%|██████████| 52.7k/52.7k [00:00<00:00, 55.3MB/s]
added_tokens.json: 100%|██████████| 34.6k/34.6k [00:00<00:00, 52.7MB/s]
special_tokens_map.json: 100%|██████████| 2.08k/2.08k [00:00<00:00, 6.14MB/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = dataset[0]["audio"]
result = pipe(sample)
print(result["text"])

Extracting data files: 100%|██████████| 1/1 [00:00<00:00, 22.53it/s]
Generating validation split: 73 examples [00:00, 9017.35 examples/s]


 Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.
